# 01. Data Exploration

In [1]:
# Import libraries and define the dataset path
import pandas as pd
from pathlib import Path

data_path = Path("../data/raw")

print([file.name for file in data_path.iterdir()])

['.gitkeep', 'movie_data.csv', 'ratings_export.csv', 'users_export.csv']


### 1. Dataset Overview

In [2]:
# Load sample records and inspect dataset columns
movies = pd.read_csv(data_path / "movie_data.csv", nrows=5)
ratings = pd.read_csv(data_path / "ratings_export.csv", nrows=5)
users = pd.read_csv(data_path / "users_export.csv", nrows=5)

print("Movies:", movies.columns.tolist())
print("Ratings:", ratings.columns.tolist())
print("Users:", users.columns.tolist())

Movies: ['_id', 'genres', 'image_url', 'imdb_id', 'imdb_link', 'movie_id', 'movie_title', 'original_language', 'overview', 'popularity', 'production_countries', 'release_date', 'runtime', 'spoken_languages', 'tmdb_id', 'tmdb_link', 'vote_average', 'vote_count', 'year_released']
Ratings: ['_id', 'movie_id', 'rating_val', 'user_id']
Users: ['_id', 'display_name', 'num_ratings_pages', 'num_reviews', 'username']


In [3]:
# Load structurally valid movie records
import csv
import re
from collections import Counter

csv.field_size_limit(2**31 - 1)

column_counts = Counter()
valid_rows = []

with open(data_path / "movie_data.csv", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    id_idx = header.index("_id")

    for row in reader:
        column_counts[len(row)] += 1

        if (
            len(row) == len(header)
            and re.fullmatch(r"[0-9a-fA-F]{24}", row[id_idx])
        ):
            valid_rows.append(row)

movies_valid = pd.DataFrame(valid_rows, columns=header)


In [4]:
# Count dataset records using previously validated Movies data and chunked reading

files = {
    "Movies": "movie_data.csv",
    "Ratings": "ratings_export.csv",
    "Users": "users_export.csv"
}

print(f"Movies (Parsed CSV Rows): {sum(column_counts.values()):,}")
print(f"Movies (Structurally Valid): {len(movies_valid):,}")

for name in ["Ratings", "Users"]:
    print(f"Reading {name}...")

    chunks = pd.read_csv(
        data_path / files[name],
        usecols=["_id"],
        chunksize=100_000,
        engine="python"
    )

    total_rows = sum(len(chunk) for chunk in chunks)
    print(f"{name}: {total_rows:,} rows")



Movies (Parsed CSV Rows): 285,963
Movies (Structurally Valid): 285,390
Reading Ratings...
Ratings: 11,078,167 rows
Reading Users...
Users: 8,139 rows


### 2. Data Structure & Types

In [5]:
# Preview sample records and inspect data types
print("MOVIES")
display(movies[["movie_id", "movie_title", "genres", "year_released"]])

print("RATINGS")
display(ratings)

print("USERS")
display(users)

print("DATA TYPES")
print("Movies:")
print(movies.dtypes)

print("\nRatings:")
print(ratings.dtypes)

print("\nUsers:")
print(users.dtypes)

MOVIES


,movie_id,movie_title,genres,year_released
0,football-freaks,Football Freaks,"[""Music"",""Animation""]",1971
1,aftermath-1960,Aftermath,[],1960
2,where-chimneys-are-seen,Where Chimneys Are Seen,"[""Drama""]",1953
3,the-musicians-daughter,The Musician's Daughter,"[""Drama""]",1911
4,50-years-of-fabulous,50 Years of Fabulous,"[""Documentary""]",2018


RATINGS


,_id,movie_id,rating_val,user_id
0,5fc57c5d6758f6963451a07f,feast-2014,7,deathproof
1,5fc57c5d6758f6963451a063,loving-2016,7,deathproof
2,5fc57c5d6758f6963451a0ef,scripted-content,7,deathproof
3,5fc57c5d6758f6963451a060,the-future,4,deathproof
4,5fc57c5c6758f69634519398,mank,5,deathproof


USERS


,_id,display_name,num_ratings_pages,num_reviews,username
0,5fc4172ec6cd28ebd99dd0e2,Lucy,32,1650,deathproof
1,5fc4172ec6cd28ebd99dd0ea,Matt Singer,52,1915,superpulse
2,5fc4172ec6cd28ebd99dd0ed,Sean Baker,21,1283,lilfilm
3,5fc4172ec6cd28ebd99dd0ee,iana,37,1177,ianamurray
4,5fc419171ebf67b9fbe48615,Lizzy,57,1810,punchdrunklizzy


DATA TYPES
Movies:
_id                         str
genres                      str
image_url                   str
imdb_id                     str
imdb_link                   str
movie_id                    str
movie_title                 str
original_language           str
overview                    str
popularity              float64
production_countries        str
release_date                str
runtime                   int64
spoken_languages            str
tmdb_id                   int64
tmdb_link                   str
vote_average            float64
vote_count                int64
year_released             int64
dtype: object

Ratings:
_id             str
movie_id        str
rating_val    int64
user_id         str
dtype: object

Users:
_id                    str
display_name           str
num_ratings_pages    int64
num_reviews          int64
username               str
dtype: object


### 3. Missing Values Analysis

In [6]:
# Recalculate missing values for structurally valid movie records

movies_missing = movies_valid.replace(
    ["", "null", "none", "nan"], pd.NA
).isna().sum()

movies_summary = pd.DataFrame({
    "Missing Count": movies_missing,
    "Missing (%)": (
        movies_missing / len(movies_valid) * 100
    ).round(2)
})

print("Movies - Missing Values:")
display(movies_summary.sort_values("Missing Count", ascending=False))


Movies - Missing Values:


,Missing Count,Missing (%)
imdb_id,41626,14.59
imdb_link,41626,14.59
overview,24256,8.50
image_url,21090,7.39
runtime,15447,5.41
release_date,14436,5.06
genres,10632,3.73
vote_count,10632,3.73
production_countries,10632,3.73
popularity,10632,3.73


In [7]:
# Analyze missing values in Ratings and Users

for name in ["Ratings", "Users"]:
    filename = files[name]
    total_rows = 0
    missing_counts = None

    chunks = pd.read_csv(
        data_path / filename,
        chunksize=100_000,
        engine="python"
    )

    for chunk in chunks:
        total_rows += len(chunk)
        current_missing = chunk.isna().sum()

        if missing_counts is None:
            missing_counts = current_missing
        else:
            missing_counts += current_missing

    missing_summary = pd.DataFrame({
        "Missing Count": missing_counts,
        "Missing (%)": (missing_counts / total_rows * 100).round(2)
    })

    print(f"\n{name} - Missing Values:")
    display(missing_summary.sort_values("Missing Count", ascending=False))


Ratings - Missing Values:


,Missing Count,Missing (%)
movie_id,6,0.0
_id,0,0.0
rating_val,0,0.0
user_id,0,0.0



Users - Missing Values:


,Missing Count,Missing (%)
num_ratings_pages,3392,41.68
display_name,307,3.77
_id,0,0.00
num_reviews,0,0.00
username,0,0.00


### 4. Duplicate Records Analysis

In [8]:
# Check duplicate identifiers in Movies and Users
users_full = pd.read_csv(
    data_path / "users_export.csv",
    engine="python"
)

for name, df, key in [
    ("Movies", movies_valid, "movie_id"),
    ("Users", users_full, "username")
]:
    print(f"\n{name} - Duplicate IDs:")
    print(f"Duplicate _id: {df['_id'].dropna().duplicated().sum():,}")
    print(f"Duplicate {key}: {df[key].dropna().duplicated().sum():,}")



Movies - Duplicate IDs:
Duplicate _id: 0
Duplicate movie_id: 0

Users - Duplicate IDs:
Duplicate _id: 0
Duplicate username: 0


In [9]:
# Check exact duplicates and inspect repeated movie identifiers

print("Movies - Exact Duplicates:", movies_valid.duplicated().sum())
print("Users - Exact Duplicates:", users_full.duplicated().sum())

duplicate_movies = movies_valid[
    (movies_valid["movie_id"].str.strip() != "")
    & movies_valid["movie_id"].duplicated(keep=False)
]

print("Movie records with repeated movie_id:", len(duplicate_movies))

display(
    duplicate_movies[
        ["_id", "movie_id", "movie_title", "year_released"]
    ].sort_values("movie_id").head(20)
)


Movies - Exact Duplicates: 0
Users - Exact Duplicates: 0
Movie records with repeated movie_id: 0


,_id,movie_id,movie_title,year_released


In [10]:
# Check duplicate rating IDs and user-movie pairs using SQLite
import sqlite3
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temp_dir:
    conn = sqlite3.connect(Path(temp_dir) / "ratings_check.db")

    conn.execute(
        "CREATE TABLE ids (id TEXT PRIMARY KEY) WITHOUT ROWID"
    )
    conn.execute(
        "CREATE TABLE pairs (user_id TEXT, movie_id TEXT, "
        "PRIMARY KEY (user_id, movie_id)) WITHOUT ROWID"
    )

    total_ids = 0
    total_pairs = 0

    chunks = pd.read_csv(
        data_path / "ratings_export.csv",
        usecols=["_id", "user_id", "movie_id"],
        chunksize=100_000,
        engine="python"
    )

    for chunk in chunks:
        ids = chunk["_id"].dropna()
        pairs = chunk[["user_id", "movie_id"]].dropna()

        total_ids += len(ids)
        total_pairs += len(pairs)

        conn.executemany(
            "INSERT OR IGNORE INTO ids VALUES (?)",
            ((value,) for value in ids)
        )

        conn.executemany(
            "INSERT OR IGNORE INTO pairs VALUES (?, ?)",
            pairs.itertuples(index=False, name=None)
        )

    unique_ids = conn.execute(
        "SELECT COUNT(*) FROM ids"
    ).fetchone()[0]

    unique_pairs = conn.execute(
        "SELECT COUNT(*) FROM pairs"
    ).fetchone()[0]

    conn.close()

print("Ratings - Duplicate Analysis:")
print(f"Duplicate _id: {total_ids - unique_ids:,}")
print(f"Duplicate user-movie pairs: {total_pairs - unique_pairs:,}")


Ratings - Duplicate Analysis:
Duplicate _id: 0
Duplicate user-movie pairs: 0


### 5. Data Validation

#### 5.1 CSV Structure Validation

In [11]:
# Report CSV structure validation results

expected_columns = len(header)
malformed_records = sum(
    count for columns, count in column_counts.items()
    if columns != expected_columns
)

print(f"Expected columns: {expected_columns}")
print(f"Column count distribution: {column_counts}")
print(f"Malformed parsed records: {malformed_records:,}")
print(f"Structurally valid movie records: {len(movies_valid):,}")


Expected columns: 19
Column count distribution: Counter({19: 285390, 1: 345, 9: 114, 11: 114})
Malformed parsed records: 573
Structurally valid movie records: 285,390


#### 5.2 Rating Validation

In [12]:
# Validate rating values and examine their distribution
rating_counts = pd.Series(dtype="int64")
total_ratings = 0
invalid_ratings = 0

chunks = pd.read_csv(
    data_path / "ratings_export.csv",
    usecols=["rating_val"],
    chunksize=100_000,
    engine="python"
)

for chunk in chunks:
    values = pd.to_numeric(chunk["rating_val"], errors="coerce")

    total_ratings += len(values)
    invalid_ratings += (~values.isin(range(1, 11))).sum()

    current_counts = values.value_counts(dropna=False)
    rating_counts = rating_counts.add(current_counts, fill_value=0)

rating_summary = pd.DataFrame({
    "Count": rating_counts.astype("int64"),
    "Percentage (%)": (rating_counts / total_ratings * 100).round(2)
}).sort_index()

print("Ratings - Value Distribution:")
display(rating_summary)

print(f"Total Ratings: {total_ratings:,}")
print(f"Invalid Ratings: {invalid_ratings:,}")


Ratings - Value Distribution:


,Count,Percentage (%)
rating_val,,
1,191316,1.73
2,383897,3.47
3,405444,3.66
4,948427,8.56
5,1113426,10.05
6,2162981,19.52
7,2033597,18.36
8,2174517,19.63
9,912069,8.23


Total Ratings: 11,078,167
Invalid Ratings: 0


#### 5.3 Release Year Validation

In [13]:
# Validate movie release years using structurally valid records

years = pd.to_numeric(
    movies_valid["year_released"],
    errors="coerce"
)

valid_years = years.dropna()

suspicious_years = valid_years[
    (valid_years < 1880) |
    (valid_years > 2026) |
    (valid_years % 1 != 0)
]

print(f"Structurally valid records checked: {len(years):,}")
print(f"Missing or non-numeric years: {years.isna().sum():,}")
print(f"Earliest year: {valid_years.min()}")
print(f"Latest year: {valid_years.max()}")
print(f"Years requiring review: {len(suspicious_years):,}")

print("\nSuspicious Year Values:")
display(suspicious_years.value_counts().sort_index())


Structurally valid records checked: 285,390
Missing or non-numeric years: 7,554
Earliest year: 1874.0
Latest year: 2028.0
Years requiring review: 23

Suspicious Year Values:


year_released
1874.0     1
1878.0    20
1879.0     1
2028.0     1
Name: count, dtype: int64

#### 5.4 Relational Integrity Validation

In [14]:
# Validate relationships between Ratings, Movies, and Users

# Collect movie IDs from structurally valid records
valid_movie_ids = set(
    movies_valid["movie_id"].loc[
        movies_valid["movie_id"].str.strip() != ""
    ]
)

# Collect user names
valid_usernames = set(users_full["username"].dropna())

# Check rating references using chunked reading
total_ratings = 0
missing_movie_ids = 0
missing_user_ids = 0
unmatched_movies = 0
unmatched_users = 0
matched_both = 0

chunks = pd.read_csv(
    data_path / "ratings_export.csv",
    usecols=["movie_id", "user_id"],
    chunksize=100_000,
    engine="python"
)

for chunk in chunks:
    total_ratings += len(chunk)

    movie_missing = chunk["movie_id"].isna()
    user_missing = chunk["user_id"].isna()

    movie_match = chunk["movie_id"].isin(valid_movie_ids)
    user_match = chunk["user_id"].isin(valid_usernames)

    missing_movie_ids += movie_missing.sum()
    missing_user_ids += user_missing.sum()

    unmatched_movies += (~movie_match & ~movie_missing).sum()
    unmatched_users += (~user_match & ~user_missing).sum()

    matched_both += (movie_match & user_match).sum()

print("Relational Integrity Results:")
print(f"Total Ratings: {total_ratings:,}")
print(f"Missing Movie IDs: {missing_movie_ids:,}")
print(f"Missing User IDs: {missing_user_ids:,}")
print(f"Unmatched Movies: {unmatched_movies:,}")
print(f"Unmatched Users: {unmatched_users:,}")
print(f"Matched Both: {matched_both:,}")
print(f"Fully Matched (%): {matched_both / total_ratings * 100:.2f}")


Relational Integrity Results:
Total Ratings: 11,078,167
Missing Movie IDs: 6
Missing User IDs: 0
Unmatched Movies: 929
Unmatched Users: 1,703
Matched Both: 11,075,529
Fully Matched (%): 99.98


#### 5.5 Genre Validation

In [15]:

# Validate movie genre formats and identify missing or empty genre lists
import json

genre_status = {
    "Valid Genres": 0,
    "Empty Genres": 0,
    "Missing Genres": 0,
    "Invalid Formats": 0
}

for genre in movies_valid["genres"]:
    genre = genre.strip()

    if genre.lower() in ("", "null", "none", "nan"):
        genre_status["Missing Genres"] += 1
        continue

    try:
        parsed_genres = json.loads(genre)

        if not isinstance(parsed_genres, list):
            genre_status["Invalid Formats"] += 1
        elif len(parsed_genres) == 0:
            genre_status["Empty Genres"] += 1
        elif all(isinstance(g, str) and g.strip() for g in parsed_genres):
            genre_status["Valid Genres"] += 1
        else:
            genre_status["Invalid Formats"] += 1

    except json.JSONDecodeError:
        genre_status["Invalid Formats"] += 1

genre_summary = pd.DataFrame(
    genre_status.items(),
    columns=["Status", "Count"]
)

total_checked = genre_summary["Count"].sum()
genre_summary["Percentage (%)"] = (
    genre_summary["Count"] / total_checked * 100
).round(2)

print(f"Movies Checked: {total_checked:,}")
display(genre_summary)


Movies Checked: 285,390


,Status,Count,Percentage (%)
0,Valid Genres,222481,77.96
1,Empty Genres,52277,18.32
2,Missing Genres,10632,3.73
3,Invalid Formats,0,0.00
